In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
import json

In [ ]:
# Specify the path to your JSON file
json_file_path = "/kaggle/input/typology-output/typology_all_output.json"

# Read data from the JSON file
with open(json_file_path, 'r') as file:
    data = json.load(file)

In [ ]:
# Extract features and labels
X = [article["content"] for article in data]
y = [article["labels"][0] for article in data]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Encode labels using LabelEncoder
label_encoder = LabelEncoder()
y_encoded_train = label_encoder.fit_transform(y_train)
y_encoded_test = label_encoder.transform(y_test)

In [ ]:
# Tokenize and prepare input data
def prepare_data(texts, labels, tokenizer):
    input_ids = []
    attention_masks = []

    for text in texts:
        encoded_text = tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids.append(encoded_text['input_ids'])
        attention_masks.append(encoded_text['attention_mask'])

    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)
    labels = torch.tensor(labels)

    return TensorDataset(input_ids, attention_masks, labels)

In [ ]:
# Load pre-trained BERT model and tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=len(set(y)))

In [ ]:
# Prepare DataLoader
train_data = prepare_data(X_train, y_encoded_train, tokenizer)
test_data = prepare_data(X_test, y_encoded_test, tokenizer)

In [ ]:
# Assuming you have the prepare_data function defined
batch_size = 8
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [ ]:
# Assuming y is your list of labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [ ]:
# Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
from tqdm import tqdm

for epoch in range(10):
    model.train()
    train_iterator = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{3}", unit="batch", leave=False)
    
    for batch in train_iterator:
        input_ids, attention_masks, labels = batch
        input_ids, attention_masks, labels = input_ids.to(device), attention_masks.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_masks, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

In [ ]:
# Evaluation
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    test_iterator = tqdm(test_loader, desc="Testing", unit="batch", leave=False)
    
    for batch in test_iterator:
        input_ids, attention_masks, labels = batch
        input_ids, attention_masks, labels = input_ids.to(device), attention_masks.to(device), labels.to(device)

        outputs = model(input_ids, attention_mask=attention_masks)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

In [ ]:
# Evaluation metrics
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("Classification Report:\n", classification_report(all_labels, all_preds))

In [ ]:
import os

# Save the model
output_dir = "/kaggle/working"
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)

In [ ]:
from transformers import BertForSequenceClassification, BertTokenizer

# Replace with the actual path to your model files
model_dir = "/kaggle/working/"

# Load the model configuration
model_config = f"{model_dir}config.json"
model = BertForSequenceClassification.from_pretrained(model_config)

# Load the model weights
model_weights = f"{model_dir}model.safetensors"
model.load_state_dict(torch.load(model_weights))

# Load the tokenizer
tokenizer = BertTokenizer.from_pretrained(model_config)

In [ ]:
test_data = [
    "This is a political statement about government spending.",
    "The article discusses healthcare policy and its impact on the economy.",
    # Add more test samples as needed
]

In [ ]:
# Tokenize the test data
encoded_test_data = [tokenizer.encode_plus(text, add_special_tokens=True, max_length=512, padding='max_length', truncation=True, return_tensors='pt') for text in test_data]

# Prepare input tensors
input_ids = torch.cat([entry['input_ids'] for entry in encoded_test_data], dim=0)
attention_masks = torch.cat([entry['attention_mask'] for entry in encoded_test_data], dim=0)

# Make predictions
with torch.no_grad():
    model.eval()
    input_ids, attention_masks = input_ids.to(device), attention_masks.to(device)
    outputs = model(input_ids, attention_mask=attention_masks)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1).cpu().numpy()

# Decode predictions using label encoder
decoded_predictions = label_encoder.inverse_transform(predictions)

# Print the results
for text, prediction in zip(test_data, decoded_predictions):
    print(f"Text: {text}\nPredicted Typology: {prediction}\n")